# 🧠 AI Logic Engine: Knowledge Ingestion (Firestore)

### 🛡️ Fix Log:
- **Fixed Header Formatting:** Removed custom metadata IDs that were causing JSON parsing errors in some mobile browsers.
- **Verified JSON:** Every brace and comma has been checked for strict standard compliance.
- **Logic Guard:** Clean ID logic remains synced with Inference Engine.

In [ ]:
# Step 1: Install & Imports
!pip install -q firebase-admin transformers accelerate bitsandbytes torch tqdm

import os, json, re, torch, time
from tqdm.auto import tqdm
import firebase_admin
from firebase_admin import credentials, firestore
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("✅ Setup Complete.")

In [ ]:
# Step 2: Firestore Connection
DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2'

if not firebase_admin._apps:
    print("Please upload your serviceAccountKey.json file:")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

db = firestore.client(database_id=DATABASE_ID)
print(f"✅ Memory Node Connected: {DATABASE_ID}")

In [ ]:
# Step 3: Normalizer
STATE_FILE = "state.json"

def clean_id(text):
    if not text: return ""
    text = text.lower().strip()
    return re.sub(r'[^a-z0-9]+', '_', text).strip('_')

def normalize_verb(v):
    is_a_set = ['is_a', 'is a', 'is type of', 'belongs to', 'category of']
    if v.strip().lower() in is_a_set: return 'is_a'
    return v.strip().lower().replace(' ', '_')

print("✅ Logic Engines Ready.")

In [ ]:
# Step 4: Load Model
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
model.eval()
print("✅ AI Model Active.")

In [ ]:
# Step 5: Start Loop
CATEGORIES = ["Biology", "Physics", "History", "Neuroscience", "Economics"]
count = 0
while True:
    try:
        cat = CATEGORIES[count % len(CATEGORIES)]
        prompt = f"List 10 facts about {cat}. Format: S|V|O|Sentence. No intro."
        messages = [{"role": "system", "content": "Output S|V|O|Sentence only."}, {"role": "user", "content": prompt}]
        input_txt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(input_txt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(inputs.input_ids, max_new_tokens=512, do_sample=True, temperature=0.7)
        resp = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        batch = db.batch()
        for line in resp.strip().split('\n'):
            parts = line.split('|')
            if len(parts) >= 3:
                s, v, o = parts[0], normalize_verb(parts[1]), parts[2]
                sid, oid = clean_id(s), clean_id(o)
                if sid and oid and sid != oid:
                    node_ref = db.collection('nodes').document(sid)
                    p = {'name': s, 'updatedAt': firestore.SERVER_TIMESTAMP}
                    if v == 'is_a': p['groups'] = firestore.ArrayUnion([oid])
                    else: p['relations'] = firestore.ArrayUnion([{'verb': v, 'targetId': oid, 'sentence': parts[3] if len(parts)>3 else '', 'weight': 100}])
                    batch.set(node_ref, p, merge=True)
        batch.commit()
        count += 1
        if count % 10 == 0: torch.cuda.empty_cache()
    except Exception as e:
        print(f"Wait: {e}")
        time.sleep(5)
    except KeyboardInterrupt: break